# Notebook 3 — Extracción de Características (Features)

Genera la matriz X y el vector y que consumirán los modelos ML.

In [1]:
import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import glob
    paths = glob.glob('/content/drive/**/emg-classification-knn-svm-ann', recursive=True)
    pc = [p for p in paths if 'Othercomputers' in p or 'Ordenadores' in p]
    PROJECT_PATH = sorted(pc or paths, key=len)[0] if paths else '/content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann'
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
except:
    if 'notebooks' in os.getcwd(): os.chdir('..')
    sys.path.insert(0, os.getcwd())

print('CWD:', os.getcwd())


Mounted at /content/drive
CWD: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann


In [2]:
from tqdm.auto import tqdm
from src.config import Config
from src.preprocessing import preprocess_pipeline, segment_gesture
from src.feature_extraction import extract_features

db_path = Config.PROCESSED_DIR / 'notebook1' / 'samples_full.pkl'
df = pd.read_pickle(db_path)
print(f'Muestras totales: {len(df)}')


Muestras totales: 18750


In [3]:
# Preprocesamiento y extracción de features por muestra
feature_rows = []
labels = []
users = []
splits = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Extrayendo features'):
    sig   = segment_gesture(row['signal_raw'], row['ground_truth_index'])
    sig_p = preprocess_pipeline(sig)
    feat  = extract_features(sig_p)  # shape (48,)
    feature_rows.append(feat)
    labels.append(row['gesture_name'])
    users.append(row['user_id'])
    splits.append(row['split'])

X = np.array(feature_rows)
y = np.array(labels)
print(f'X shape: {X.shape}  |  y shape: {y.shape}')


Extrayendo features:   0%|          | 0/18750 [00:00<?, ?it/s]

X shape: (18750, 48)  |  y shape: (18750,)


In [4]:
# Guardar
out_dir = Config.PROCESSED_DIR / 'features'
os.makedirs(out_dir, exist_ok=True)
np.savez_compressed(out_dir / 'emg_features_base.npz', X=X, y=y)
np.savez_compressed(out_dir / 'emg_metadata_base.npz', y=y, users=np.array(users), splits=np.array(splits))
print(f'Guardado en: {out_dir}')


Guardado en: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann/data/processed/features
